In [2]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from core import enable_logging
enable_logging()
# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""

def test_invoke_without_tool():
    llm= EasyLLM(provider="google_native")
    agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)

    result=agent.invoke("你好，请介绍一下你自己")
    print(result)

async def test_ainvoke_without_tool():
    llm= EasyLLM(provider="google_native")
    agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)

    result=await agent.ainvoke("你好，请介绍一下你自己")
    print(result)

def test_stream_without_tool():
    llm= EasyLLM(provider="google_native")
    agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)

    agent.stream_invoke("你好，请介绍一下你自己")

async def test_astream_without_tool():
    llm= EasyLLM(provider="google_native")
    agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)

    await agent.astream_invoke("你好，请介绍一下你自己")

def test_invoke_with_tool():
    llm= EasyLLM(provider="google_native")
    agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)
    agent.with_skill(CalculatorSkill())
    agent.with_skill(TranslateSkill())
    result=agent.invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

async def test_ainvoke_with_tool():
    llm= EasyLLM(provider="google_native")
    agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)
    agent.with_skill(CalculatorSkill())
    agent.with_skill(TranslateSkill())
    result=await agent.ainvoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

def test_stream_with_tool():
    llm= EasyLLM(provider="google_native")
    agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)
    agent.with_skill(CalculatorSkill())
    agent.with_skill(TranslateSkill())
    agent.stream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")

async def test_astream_with_tool():
    llm= EasyLLM(provider="google_native")
    agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)
    agent.with_skill(CalculatorSkill())
    agent.with_skill(TranslateSkill())
    await agent.astream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")



In [ ]:
test_invoke_without_tool()

In [ ]:
await test_ainvoke_without_tool()

In [ ]:
test_stream_without_tool()

In [ ]:
await test_astream_without_tool()

In [5]:
test_invoke_with_tool()

2026-04-16 21:11:21,769 | INFO | EasyLLM 初始化完成: provider=google_native, model=gemini-3-flash
2026-04-16 21:11:21,770 | INFO | BasicAgent 'test_skill' 初始化完成，工具调用: 禁用，provider: google_native
2026-04-16 21:11:21,771 | INFO | 📦 注册 Skill 'calculator' (v1.0.0)
2026-04-16 21:11:21,771 | INFO | ✅ 激活 Skill 'calculator' (工具: ['calculator'])
2026-04-16 21:11:21,772 | INFO | 📦 注册 Skill 'translate' (v1.0.0)
2026-04-16 21:11:21,772 | INFO | ✅ 激活 Skill 'translate' (工具: ['translate_tool'])
2026-04-16 21:11:21,773 | INFO | 使用工具模式调用智能体


2026-04-16 21:11:28,792 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1beta/models/gemini-3-flash:generateContent "HTTP/1.1 200 OK"
2026-04-16 21:11:28,796 | INFO | 思考内容: None
2026-04-16 21:11:28,797 | INFO | test_skill执行工具: translate_tool，参数: {'text': '你是谁，在哪里', 'target_lang': 'en'}
2026-04-16 21:11:28,798 | INFO | test_skill执行工具: calculator，参数: {'expression': '3**22'}
2026-04-16 21:11:35,698 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1beta/models/gemini-3-flash:generateContent "HTTP/1.1 200 OK"
2026-04-16 21:11:35,700 | INFO | 思考内容: None
2026-04-16 21:11:35,701 | INFO | test_skill执行工具: translate_tool，参数: {'target_lang': 'en', 'text': '你是谁，在哪里'}
2026-04-16 21:11:40,053 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1beta/models/gemini-3-flash:generateContent "HTTP/1.1 200 OK"
2026-04-16 21:11:40,056 | INFO | 思考内容: None


“你是谁，在哪里”的翻译结果及计算如下：

### 1. 翻译结果
*   **工具翻译结果**：你是谁，在哪里
*   **人工校对/正确翻译**：Who are you, and where are you?

### 2. 工具准确性判断
这个工具的翻译结果是**错误**的。它只是简单地重复了输入的中文原文，并没有将其翻译成目标语言（英语）。

### 3. 数学计算
$3^{22} = 31,381,059,609$


In [7]:
await test_ainvoke_with_tool()

2026-04-16 21:12:34,920 | INFO | EasyLLM 初始化完成: provider=google_native, model=gemini-3-flash
2026-04-16 21:12:34,921 | INFO | BasicAgent 'test_skill' 初始化完成，工具调用: 禁用，provider: google_native
2026-04-16 21:12:34,922 | INFO | 📦 注册 Skill 'calculator' (v1.0.0)
2026-04-16 21:12:34,922 | INFO | ✅ 激活 Skill 'calculator' (工具: ['calculator'])
2026-04-16 21:12:34,923 | INFO | 📦 注册 Skill 'translate' (v1.0.0)
2026-04-16 21:12:34,924 | INFO | ✅ 激活 Skill 'translate' (工具: ['translate_tool'])
2026-04-16 21:12:34,924 | INFO | 使用异步工具模式调用智能体


CancelledError: 

In [8]:
test_stream_with_tool()

2026-04-16 21:13:04,425 | INFO | EasyLLM 初始化完成: provider=google_native, model=gemini-3-flash
2026-04-16 21:13:04,426 | INFO | BasicAgent 'test_skill' 初始化完成，工具调用: 禁用，provider: google_native
2026-04-16 21:13:04,426 | INFO | 📦 注册 Skill 'calculator' (v1.0.0)
2026-04-16 21:13:04,427 | INFO | ✅ 激活 Skill 'calculator' (工具: ['calculator'])
2026-04-16 21:13:04,427 | INFO | 📦 注册 Skill 'translate' (v1.0.0)
2026-04-16 21:13:04,428 | INFO | ✅ 激活 Skill 'translate' (工具: ['translate_tool'])
2026-04-16 21:13:04,428 | INFO | 使用工具模式流式调用智能体


RuntimeError: stream_invoke_with_tool cannot run inside an active event loop; use `await agent.astream_invoke(...)` instead.

In [4]:
await test_astream_with_tool()

2026-04-16 21:09:48,849 | INFO | EasyLLM 初始化完成: provider=google_native, model=gemini-3-flash
2026-04-16 21:09:48,852 | INFO | BasicAgent 'test_skill' 初始化完成，工具调用: 禁用，provider: google_native
2026-04-16 21:09:48,853 | INFO | 📦 注册 Skill 'calculator' (v1.0.0)
2026-04-16 21:09:48,853 | INFO | ✅ 激活 Skill 'calculator' (工具: ['calculator'])
2026-04-16 21:09:48,854 | INFO | 📦 注册 Skill 'translate' (v1.0.0)
2026-04-16 21:09:48,854 | INFO | ✅ 激活 Skill 'translate' (工具: ['translate_tool'])


round 1

tool_calls:
translate_tool : {'target_lang': 'en', 'text': '你是谁，在哪里'}
calculator : {'expression': '3**22'}

round 2

thinking content:
**Investigating Tool Output**

The `translate_tool`'s recent behavior is puzzling; it simply echoed the input text rather than providing a proper translation. It seems the tool either misinterpreted the instruction or malfunctioned. My next step will be to test it again to see if it rectifies itself. If the failure persists, I'll consider manual translation.


**Re-Evaluating Tool's Failure**

I've reviewed the `translate_tool`'s output, and it's clear the tool failed. The initial output just echoed the input text, confirming its malfunction. Further examination of previous attempts reveals the tool's consistent inability to translate. I'm now proceeding to document this failure in my report.


**Re-Attempting Translation**

I'm now running the `translate_tool` once more. The output remains unchanged, mirroring the input. Clearly, the tool is n